# Assignment 1

### Group Members

### Installation (Only Run if Needed)

In [ ]:
%pip install pymongo

### Import Packages

In [1]:
from pymongo import MongoClient, ReadPreference
from pymongo.write_concern import WriteConcern
from pymongo.read_concern import ReadConcern

import pymongo
import time

### Default Connections

In [ ]:
uri = "mongodb://localhost:27017/"
client = MongoClient(uri)

In [ ]:
db = client["university"]
collection = db["students"]

In [ ]:
# Define the query filter
query = {"student_id": "S007"}

# Retrieve the first matching document
result = collection.find_one(query)
result

### Replica Set Connection

In [2]:
client = MongoClient(
    "mongodb://mongo1:27017,mongo2:27017,mongo3:27017/?replicaSet=dsa5208",
    serverSelectionTimeoutMS=5000,
    w=1,
    readConcernLevel="local"
)

In [3]:
hello = client.admin.command("hello")

print("Replica set:", hello.get("setName"))
print("Primary:", hello.get("primary"))
print("Writable primary:", hello.get("isWritablePrimary"))

Replica set: dsa5208
Primary: mongo1:27017
Writable primary: True


In [4]:
for name in ["mongo1", "mongo2", "mongo3"]:
    try:
        c = MongoClient(f"mongodb://{name}:27017/?directConnection=true", 
                        serverSelectionTimeoutMS=5000)
        info = c.admin.command("hello")
        print(name, "is connected to:", info.get("me"))
    except Exception as e:
        print(name, "FAILED:", e)

mongo1 is connected to: mongo1:27017
mongo2 is connected to: mongo2:27017
mongo3 is connected to: mongo3:27017


In [5]:
db = client.get_database('university', 
                         read_preference=ReadPreference.SECONDARY
                        )

#### Read Follow Writes

In [6]:
students = db["students"]

'''
for i in range(999):
    students.insert_one({
            "student_id": f"S{i:03}",
            "student_name": f"Robert Tan - {i}",
            "student_status": "Graduate",
            "senority" : f'Year {i%4}'
            "academic_load": "Full Time"
    })
'''

students.insert_one({
        "student_id": "S001",
        "student_name": "Robert Tan",
        "student_status": "Undergraduate",
        "senority" : "Year 1",
        "academic_load": "Full Time"
})

InsertOneResult(ObjectId('6a9bd9d82616499e4572ccfd'), acknowledged=True)

In [7]:
db.students.find_one({"student_id": "S001"})

{'_id': ObjectId('6a9bd9d82616499e4572ccfd'),
 'student_id': 'S001',
 'student_name': 'Robert Tan',
 'student_status': 'Undergraduate',
 'senority': 'Year 1',
 'academic_load': 'Full Time'}

In [8]:
client3 = MongoClient(
    "mongodb://mongo3:27017/?directConnection=true",
    serverSelectionTimeoutMS=3000
)

In [9]:
db3 = client3.get_database('university')

In [10]:
db3.students.find_one({"student_id": "S001"})

{'_id': ObjectId('6a9bd9d82616499e4572ccfd'),
 'student_id': 'S001',
 'student_name': 'Robert Tan',
 'student_status': 'Undergraduate',
 'senority': 'Year 1',
 'academic_load': 'Full Time'}

In [11]:
client3.admin.command({
    "configureFailPoint": "rsSyncApplyStop",
    "mode": "alwaysOn"
})

{'count': 52,
 'ok': 1.0,
 '$clusterTime': {'clusterTime': Timestamp(1788598744, 1),
  'signature': {'hash': b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00',
   'keyId': 0}},
 'operationTime': Timestamp(1788598744, 1)}

In [13]:
db.students.update_one(
   { "student_id": "S001"},         
   { "$set" : {"student_status": "Graduate"} }
)

db.students.find_one({"student_id": "S001"})

{'_id': ObjectId('6a9bd9d82616499e4572ccfd'),
 'student_id': 'S001',
 'student_name': 'Robert Tan',
 'student_status': 'Undergraduate',
 'senority': 'Year 1',
 'academic_load': 'Full Time'}

In [14]:
db3.students.find_one({"student_id": "S001"})

{'_id': ObjectId('6a9bd9d82616499e4572ccfd'),
 'student_id': 'S001',
 'student_name': 'Robert Tan',
 'student_status': 'Undergraduate',
 'senority': 'Year 1',
 'academic_load': 'Full Time'}

In [21]:
status = client.admin.command("replSetGetStatus")

for m in status["members"]:
    print(m["name"], m["stateStr"], m.get("optimeDate"))

mongo1:27017 PRIMARY 2026-09-05 09:00:59
mongo2:27017 SECONDARY 2026-09-05 09:00:59
mongo3:27017 SECONDARY 2026-09-05 08:59:04


In [22]:
client3.admin.command({
    "configureFailPoint": "rsSyncApplyStop",
    "mode": "off"
})

db3.students.find_one({"student_id": "S001"})

{'_id': ObjectId('6a9bd9d82616499e4572ccfd'),
 'student_id': 'S001',
 'student_name': 'Robert Tan',
 'student_status': 'Undergraduate',
 'senority': 'Year 1',
 'academic_load': 'Full Time'}

In [23]:
with client.start_session(causal_consistency=True) as session:
    db.students.update_one(
        {"student_id": "S001"},
        {"$set": {"student_status": "Post Doc"}},
        session=session
    )
    
    doc = db.students.find_one({"student_id": "S001"}, session=session)
    print("With session:", doc)

With session: {'_id': ObjectId('6a9bd9d82616499e4572ccfd'), 'student_id': 'S001', 'student_name': 'Robert Tan', 'student_status': 'Post Doc', 'senority': 'Year 1', 'academic_load': 'Full Time'}


In [25]:
status = client.admin.command("replSetGetStatus")

for m in status["members"]:
    print(m["name"], m["stateStr"], m.get("optimeDate"))

mongo1:27017 PRIMARY 2026-09-05 09:01:11
mongo2:27017 SECONDARY 2026-09-05 09:01:11
mongo3:27017 SECONDARY 2026-09-05 09:01:11


#### Monotonic Reads

In [26]:
students.insert_one({
        "student_id": "S002",
        "student_name": "Feyd Johnston",
        "student_status": "Undergraduate",
        "senority" : "Year 1",
        "academic_load": "Full Time"
})

InsertOneResult(ObjectId('6a9bda5b2616499e4572ccff'), acknowledged=True)

In [27]:
db.students.find_one({"student_id": "S002"})

{'_id': ObjectId('6a9bda5b2616499e4572ccff'),
 'student_id': 'S002',
 'student_name': 'Feyd Johnston',
 'student_status': 'Undergraduate',
 'senority': 'Year 1',
 'academic_load': 'Full Time'}

In [28]:
client3.admin.command({
    "configureFailPoint": "rsSyncApplyStop",
    "mode": "alwaysOn"
})

{'count': 54,
 'ok': 1.0,
 '$clusterTime': {'clusterTime': Timestamp(1788598875, 1),
  'signature': {'hash': b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00',
   'keyId': 0}},
 'operationTime': Timestamp(1788598875, 1)}

In [29]:
for i in range(2, 6):
    db.students.update_one(
        {"student_id": "S002"},
        {"$set": {"senority": f"Year {i}"}}
    )

In [31]:
no_session_reads = []
for _ in range(10):
    doc = db.students.with_options(read_preference=ReadPreference.SECONDARY).find_one(
        {"student_id": "S002"}
    )
    no_session_reads.append(doc["senority"])

print("No session:", no_session_reads)

No session: ['Year 5', 'Year 5', 'Year 1', 'Year 5', 'Year 5', 'Year 5', 'Year 5', 'Year 5', 'Year 1', 'Year 1']


In [32]:
with client.start_session(causal_consistency=True) as session:
    with_session_reads = []
    
    for _ in range(10):
        try:
            doc = db.students.with_options(read_preference=ReadPreference.SECONDARY).find_one(
                {"student_id": "S002"}, session=session, max_time_ms=3000
            )
            with_session_reads.append(doc["senority"])
        except Exception as e:
            with_session_reads.append("TIMEOUT")

print("With session:", with_session_reads)

With session: ['Year 5', 'TIMEOUT', 'TIMEOUT', 'Year 5', 'Year 5', 'Year 5', 'Year 5', 'Year 5', 'TIMEOUT', 'TIMEOUT']


In [33]:
client3.admin.command({
    "configureFailPoint": "rsSyncApplyStop",
    "mode": "off"
})

{'count': 56,
 'ok': 1.0,
 '$clusterTime': {'clusterTime': Timestamp(1788598909, 1),
  'signature': {'hash': b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00',
   'keyId': 0}},
 'operationTime': Timestamp(1788598875, 1)}

#### Monotonic Write

In [34]:
students.insert_one({
        "student_id": "S003",
        "student_name": "Alan White",
        "student_status": "Post Doc",
        "senority" : "Year 1",
        "academic_load": "Full Time"
})

InsertOneResult(ObjectId('6a9bda802616499e4572cd00'), acknowledged=True)

In [35]:
db3.students.find_one({"student_id": "S003"})

{'_id': ObjectId('6a9bda802616499e4572cd00'),
 'student_id': 'S003',
 'student_name': 'Alan White',
 'student_status': 'Post Doc',
 'senority': 'Year 1',
 'academic_load': 'Full Time'}

In [36]:
db.students.update_one(
    {"student_id": "S003"},
    {"$set": {"academic_history": ["B.Sc. Computer Science"]}},
    upsert=True
)

UpdateResult({'n': 1, 'electionId': ObjectId('7fffffff0000000000000003'), 'opTime': {'ts': Timestamp(1788598917, 1), 't': 3}, 'nModified': 1, 'ok': 1.0, '$clusterTime': {'clusterTime': Timestamp(1788598917, 1), 'signature': {'hash': b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00', 'keyId': 0}}, 'operationTime': Timestamp(1788598917, 1), 'updatedExisting': True}, acknowledged=True)

In [37]:
client3.admin.command({
    "configureFailPoint": "rsSyncApplyStop",
    "mode": "alwaysOn"
})

{'count': 56,
 'ok': 1.0,
 '$clusterTime': {'clusterTime': Timestamp(1788598917, 1),
  'signature': {'hash': b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00',
   'keyId': 0}},
 'operationTime': Timestamp(1788598917, 1)}

In [38]:
db.students.update_one({"student_id": "S003"}, {"$push": {"academic_history": "M.Sc. Data Science"}})
db.students.update_one({"student_id": "S003"}, {"$push": {"academic_history": "PhD Information Systems"}})

UpdateResult({'n': 1, 'electionId': ObjectId('7fffffff0000000000000003'), 'opTime': {'ts': Timestamp(1788598921, 2), 't': 3}, 'nModified': 1, 'ok': 1.0, '$clusterTime': {'clusterTime': Timestamp(1788598921, 2), 'signature': {'hash': b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00', 'keyId': 0}}, 'operationTime': Timestamp(1788598921, 2), 'updatedExisting': True}, acknowledged=True)

In [39]:
print("mongo3 mid-lag:", db3.students.find_one({"student_id": "S003"})["academic_history"])

mongo3 mid-lag: ['B.Sc. Computer Science']


In [40]:
db.students.find_one({"student_id": "S003"})["academic_history"]

['B.Sc. Computer Science']

In [42]:
results = []
for _ in range(20):
    doc = db.students.find_one({"student_id": "S003"})
    results.append(doc["academic_history"])

for r in results:
    print(r)

['B.Sc. Computer Science']
['B.Sc. Computer Science']
['B.Sc. Computer Science', 'M.Sc. Data Science', 'PhD Information Systems']
['B.Sc. Computer Science', 'M.Sc. Data Science', 'PhD Information Systems']
['B.Sc. Computer Science']
['B.Sc. Computer Science']
['B.Sc. Computer Science']
['B.Sc. Computer Science', 'M.Sc. Data Science', 'PhD Information Systems']
['B.Sc. Computer Science']
['B.Sc. Computer Science']
['B.Sc. Computer Science']
['B.Sc. Computer Science', 'M.Sc. Data Science', 'PhD Information Systems']
['B.Sc. Computer Science']
['B.Sc. Computer Science']
['B.Sc. Computer Science', 'M.Sc. Data Science', 'PhD Information Systems']
['B.Sc. Computer Science', 'M.Sc. Data Science', 'PhD Information Systems']
['B.Sc. Computer Science', 'M.Sc. Data Science', 'PhD Information Systems']
['B.Sc. Computer Science', 'M.Sc. Data Science', 'PhD Information Systems']
['B.Sc. Computer Science']
['B.Sc. Computer Science', 'M.Sc. Data Science', 'PhD Information Systems']


In [43]:
client3.admin.command({
    "configureFailPoint": "rsSyncApplyStop",
    "mode": "off"
})

time.sleep(2)

print("mongo3 (after sync is enabled):", db3.students.find_one({"student_id": "S003"})["academic_history"])

mongo3 (after sync is enabled): ['B.Sc. Computer Science', 'M.Sc. Data Science', 'PhD Information Systems']


#### Write Follows Read

In [44]:
courses = db["courses"]

courses.insert_one({
        "course_id": "C001",
        "course_name": "Programming Methodology",
        "course_status": "ACTIVE",
        "vacancy" : "30"
})

InsertOneResult(ObjectId('6a9bda9e2616499e4572cd01'), acknowledged=True)

In [45]:
db.courses.find_one({"course_id": "C001"})

{'_id': ObjectId('6a9bda9e2616499e4572cd01'),
 'course_id': 'C001',
 'course_name': 'Programming Methodology',
 'course_status': 'ACTIVE',
 'vacancy': '30'}

In [46]:
db3.courses.find_one({"course_id": "C001"})

{'_id': ObjectId('6a9bda9e2616499e4572cd01'),
 'course_id': 'C001',
 'course_name': 'Programming Methodology',
 'course_status': 'ACTIVE',
 'vacancy': '30'}

In [47]:
students.update_one(
    {"student_id": "S002"},
    {"$set": {"enrollment_note": None}}
)

UpdateResult({'n': 1, 'electionId': ObjectId('7fffffff0000000000000003'), 'opTime': {'ts': Timestamp(1788598954, 1), 't': 3}, 'nModified': 1, 'ok': 1.0, '$clusterTime': {'clusterTime': Timestamp(1788598954, 1), 'signature': {'hash': b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00', 'keyId': 0}}, 'operationTime': Timestamp(1788598954, 1), 'updatedExisting': True}, acknowledged=True)

In [48]:
client3.admin.command({
    "configureFailPoint": "rsSyncApplyStop",
    "mode": "alwaysOn"
})

{'count': 58,
 'ok': 1.0,
 '$clusterTime': {'clusterTime': Timestamp(1788598954, 1),
  'signature': {'hash': b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00',
   'keyId': 0}},
 'operationTime': Timestamp(1788598954, 1)}

In [49]:
course = db.courses.with_options(read_preference=ReadPreference.SECONDARY).find_one({"course_id": "C001"})
print("Read course ", course["course_name"], " with status:", course["course_status"])

db.students.update_one(
    {"student_id": "S002"},
    {"$set": {"enrollment_note": f"Enrolled in {course['course_name']} while course_status was: {course['course_status']}"}}
)

Read course  Programming Methodology  with status: ACTIVE


UpdateResult({'n': 1, 'electionId': ObjectId('7fffffff0000000000000003'), 'opTime': {'ts': Timestamp(1788598958, 1), 't': 3}, 'nModified': 1, 'ok': 1.0, '$clusterTime': {'clusterTime': Timestamp(1788598958, 1), 'signature': {'hash': b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00', 'keyId': 0}}, 'operationTime': Timestamp(1788598958, 1), 'updatedExisting': True}, acknowledged=True)

In [50]:
db3.courses.find_one({"course_id": "C001"})

{'_id': ObjectId('6a9bda9e2616499e4572cd01'),
 'course_id': 'C001',
 'course_name': 'Programming Methodology',
 'course_status': 'ACTIVE',
 'vacancy': '30'}

In [51]:
db3.students.find_one({"student_id": "S002"})

{'_id': ObjectId('6a9bda5b2616499e4572ccff'),
 'student_id': 'S002',
 'student_name': 'Feyd Johnston',
 'student_status': 'Undergraduate',
 'senority': 'Year 5',
 'academic_load': 'Full Time',
 'enrollment_note': None}

In [53]:
db.students.find_one({"student_id": "S002"})

{'_id': ObjectId('6a9bda5b2616499e4572ccff'),
 'student_id': 'S002',
 'student_name': 'Feyd Johnston',
 'student_status': 'Undergraduate',
 'senority': 'Year 5',
 'academic_load': 'Full Time',
 'enrollment_note': 'Enrolled in Programming Methodology while course_status was: ACTIVE'}

In [55]:
with client.start_session(causal_consistency=True) as session:
    course = db.courses.with_options(read_preference=ReadPreference.SECONDARY).find_one(
        {"course_id": "C001"}, session=session
    )
    print("Session read course status:", course["course_status"])

    db.students.update_one(
        {"student_id": "S002"},
        {"$set": {"enrollment_note": f"Session enrollment, course_status: {course['course_status']}"}},
        session=session
    )

    try:
        doc = db.students.with_options(read_preference=ReadPreference.SECONDARY).find_one(
            {"student_id": "S002"}, session=session, max_time_ms=3000
        )
        print("Session read-back:", doc)
    except Exception as e:
        print("Timed out:", e)

Session read course status: ACTIVE
Session read-back: {'_id': ObjectId('6a9bda5b2616499e4572ccff'), 'student_id': 'S002', 'student_name': 'Feyd Johnston', 'student_status': 'Undergraduate', 'senority': 'Year 5', 'academic_load': 'Full Time', 'enrollment_note': 'Session enrollment, course_status: ACTIVE'}


In [ ]:
client3.admin.command({
    "configureFailPoint": "rsSyncApplyStop",
    "mode": "off"
})